    """
    Calculate Counts Per Million (CPM) for each column (sample) in a count matrix.

    Parameters
    ----------
    count_matrix : numpy.ndarray or pandas.DataFrame
       2D array where rows are features (genes) and columns are samples.
       Can be a NumPy array or a pandas DataFrame.

    Returns
    -------
    cpm_matrix : numpy.ndarray
       CPM-normalized values in the same shape as input.
    """

In [ ]:
import pandas as pd
import scipy.stats as stats
import numpy as np

def calculate_cpm(count_matrix):

    # If input is DataFrame, convert to numpy array
    if hasattr(count_matrix, 'values'):
        counts = count_matrix.values
    else:
        counts = np.array(count_matrix)

    # Sum counts per sample (column sum)
    col_sums = counts.sum(axis=0)

    # Avoid division by zero (replace zeros with 1 to prevent inf, or handle as needed)
    col_sums[col_sums == 0] = 1

    # Calculate CPM
    cpm_matrix = (counts / col_sums) * 1e6

    return cpm_matrix

df = pd.read_csv("merged_featureCounts.txt", sep = "\t", index_col=0)

df_cpm = calculate_cpm(df)
df_cpm = np.log(df_cpm + 1)
df_cpm = pd.DataFrame(df_cpm, index=df.index, columns=df.columns)

#parsing sample type column
#meta_data = pd.DataFrame({"Sample_ID" : df.columns, "condition" : df.columns}, index=df.columns)
#meta_data["condition"] = [list(x.split("_")[0])[-1] for x in meta_data.condition]
#meta_data = meta_data.loc[meta_data.condition == "M"]

# from reference: samples '7M_RCS' and '19.2M_Pitt' were dropped, the first due to a lack of matching clinical data and the second due to sample replication
#sample_of_interest = [x for x in meta_data.sample_id if x not in ["7M_RCS", "19-2M_Pitt"]]
#meta_data_subset = meta_data.loc[meta_data.sample_id.isin(sample_of_interest)]

clinical_data = pd.read_csv("clinical_data.txt", sep = "\t")
#clinical_data = clinical_data.loc[clinical_data.sample_id != "19_2_Pitt"]
clinical_data.index = list(clinical_data.Sample_ID)

gene_symbols = [
    "ENSG00000100146",  # SOX10
    "ENSG00000205927",  # OLIG2
    "ENSG00000184221",  # OLIG1
    "ENSG00000134853",  # PDGFRA
    "ENSG00000173546",  # CSPG4
    "ENSG00000106278",  # PTPRZ1
    "ENSG00000144230",  # GPR17



    ]
"""
"ENSG00000196136",  # SERPINA3
    "ENSG00000224389"   # C4B
gene_symbols = ["ENSG00000101144"]
"""
df_subset = df_cpm.loc[df_cpm.index.isin(gene_symbols)]

## subset of gene expression for sample
df_subset = df_subset[df_cpm.Sample_ID]
df_subset.to_csv("exp_data.csv")

#meta_data_subset["sample_id"] = [x.replace("M", "") for x in meta_data_subset.sample_id]
df_cpm.index = list(df_cpm.Sample_ID)
surv_data = pd.merge(df_cpm, clinical_data, left_index=True, right_index=True)
surv_data.to_csv("meta_data.csv")

if(len(gene_symbols) > 1):
    # Reference: the sum of the scaled data (z-scores) for multigene signatures. NOTE: this is for the gene signature and not for the individual gene.
    df_subset_z_score = stats.zscore(df_subset)
    df_subset_z_score = pd.DataFrame(df_subset_z_score, index=df_subset.index, columns=df_subset.columns)
    df_subset_z_score.to_csv("zscore.csv")
    surv_data["score"] = np.array(df_subset_z_score.sum())
else:
    surv_data["score"] = np.array(df_subset.sum())

median_score = np.median(surv_data.score)

surv_data["group"] = np.array(["high" if x > median_score else "low" for x in surv_data.score])
surv_data.to_csv("surv_data.csv")
surv_data = surv_data[['Sample_ID_x'  #'condition'
                        , 'Sample_ID_y', 'Vital_Status'# 'DFS', 'BMSF', 'SPBM',
                        'OS', 'group']]

FileNotFoundError: [Errno 2] No such file or directory: 'clinical_data.txt'

In [ ]:
!pip install lifelines

In [ ]:
import re
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter, KaplanMeierFitter, statistics
import matplotlib.pyplot as plt

# ---------- 1. CPM-TMM Normalization ----------
def calculate_cpm_tmm(count_matrix, ref_col=None, logratio_trim=0.3, sum_trim=0.05):
    """
    Calculate CPM normalized with TMM (Trimmed Mean of M-values).
    count_matrix: DataFrame (genes x samples) of raw counts
    ref_col: reference sample (default = sample with median library size)
    logratio_trim: fraction to trim from M-values
    sum_trim: fraction to trim from A-values
    """
    if not isinstance(count_matrix, pd.DataFrame):
        raise ValueError("Input must be a pandas DataFrame with genes as rows, samples as columns")

    counts = count_matrix.astype(float)
    lib_sizes = counts.sum(axis=0)

    # choose reference sample = median library size if not specified
    if ref_col is None:
        ref_col = lib_sizes.index[np.argsort(lib_sizes.values)[len(lib_sizes)//2]]

    ref_counts = counts[ref_col]
    ref_libsize = lib_sizes[ref_col]

    norm_factors = {}

    for col in counts.columns:
        if col == ref_col:
            norm_factors[col] = 1.0
            continue

        y = counts[col]
        libsize = lib_sizes[col]

        # avoid division by zero
        valid = (ref_counts > 0) & (y > 0)
        if valid.sum() == 0:
            norm_factors[col] = 1.0
            continue

        ref_y = ref_counts[valid]
        y = y[valid]

        # log ratios (M-values) and absolute expression (A-values)
        logR = np.log2((y / libsize) / (ref_y / ref_libsize))
        absA = 0.5 * np.log2((y / libsize) * (ref_y / ref_libsize))

        # weights
        w = 1 / ((1/y) + (1/ref_y))

        # trimming
        n = len(logR)
        loL = int(n * logratio_trim / 2)
        hiL = n - loL
        loS = int(n * sum_trim / 2)
        hiS = n - loS

        keep = np.argsort(logR)[loL:hiL]
        keep = np.intersect1d(keep, np.argsort(absA)[loS:hiS])

        if len(keep) == 0:
            norm_factors[col] = 1.0
        else:
            norm_factors[col] = 2 ** (np.sum(w[keep] * logR[keep]) / np.sum(w[keep]))

    # scale so geometric mean = 1
    nf_series = pd.Series(norm_factors)
    nf_series = nf_series / np.exp(np.mean(np.log(nf_series)))

    # compute CPM with TMM factors
    cpm = (counts / lib_sizes) * 1e6
    cpm_tmm = cpm / nf_series

    return cpm_tmm


# ---------- 2. Load Expression Data ----------
df = pd.read_csv("merged_featureCounts.txt", sep="\t", index_col=0)

# CPM-TMM + log-transform
df_cpm = calculate_cpm_tmm(df)
df_cpm = np.log(df_cpm + 1)
df_cpm = pd.DataFrame(df_cpm, index=df.index, columns=df.columns)


# ---------- 3. Metadata Cleaning ----------
meta_data = pd.DataFrame({"sample_id": df.columns}, index=df.columns)

def clean_name(x):
    s = x.replace("-", "_")
    s = re.sub(r'(?<=\d)M', '', s)
    s = re.sub(r'__+', '_', s)
    return s

meta_data["sample_id_clean"] = meta_data["sample_id"].astype(str).map(clean_name)

def detect_condition(x):
    parts = x.split("_")
    first = parts[0]
    if len(first) >= 1:
        last = first[-1]
        if last.isalpha():
            return last
    return np.nan

meta_data["condition"] = meta_data["sample_id"].map(detect_condition)

# keep only condition == "M"
meta_data = meta_data.loc[meta_data["condition"] == "M"].copy()

# drop problematic samples
exclude_manual = {"7M_RCS", "19-2M_Pitt"}
meta_data = meta_data.loc[~meta_data["sample_id"].isin(exclude_manual)].copy()

sample_map = dict(zip(meta_data["sample_id"], meta_data["sample_id_clean"]))


# ---------- 4. Clinical Data ----------
clinical_data = pd.read_csv("clinical_data.txt", sep="\t")
clinical_data["sample_id"] = clinical_data["sample_id"].astype(str).map(lambda x: clean_name(x))
clinical_data = clinical_data.set_index("sample_id", drop=False)

keep_clin = clinical_data.index.intersection(meta_data["sample_id_clean"])
clinical_data = clinical_data.loc[keep_clin].copy()
if clinical_data.shape[0] == 0:
    raise ValueError("No clinical samples matched metadata after cleaning. Check sample naming conventions.")


# ---------- 5. Genes of Interest ----------
gene_symbols = [
    "ENSG00000100146",  # SOX10
    "ENSG00000205927",  # OLIG2
    "ENSG00000184221",  # OLIG1
    "ENSG00000134853",  # PDGFRA
    "ENSG00000173546",  # CSPG4
    "ENSG00000106278",  # PTPRZ1
    "ENSG00000144230",  # GPR17

]

df_subset = df_cpm.loc[df_cpm.index.isin(gene_symbols)].copy()
available_samples = [s for s in meta_data["sample_id"] if s in df_subset.columns]
df_subset = df_subset[available_samples].copy()
rename_map = {orig: sample_map[orig] for orig in available_samples}
df_subset = df_subset.rename(columns=rename_map)


# ---------- 6. Merge Metadata + Clinical ----------
meta_small = meta_data.set_index("sample_id_clean", drop=False).loc[df_subset.columns].copy()
surv_data = meta_small.merge(clinical_data, left_index=True, right_index=True, how="inner", suffixes=("_meta", "_clin"))

if "Status" not in surv_data.columns:
    raise KeyError("clinical_data must contain a 'Status' column (Alive/Dead).")
surv_data['Status'] = surv_data['Status'].map({'Alive': 0, 'Dead': 1})
surv_data = surv_data.dropna(subset=["OS", "Status"])
surv_data["OS"] = pd.to_numeric(surv_data["OS"], errors="coerce")
surv_data = surv_data.dropna(subset=["OS"])
final_samples = surv_data.index.tolist()


# ---------- 7. Z-score Normalization ----------
df_subset = df_subset.loc[:, df_subset.columns.intersection(final_samples)].copy()
df_norm = df_subset.apply(lambda x: (x - x.mean()) / (x.std() if x.std() != 0 else np.nan), axis=1)
df_norm = df_norm.dropna(how='all')


# ---------- 8. Cox Regression per Gene ----------
weights = {}
for gene in df_norm.index:
    cph = CoxPHFitter()
    expr_series = df_norm.loc[gene].reindex(surv_data.index)
    tmp = pd.DataFrame({
        "expr": expr_series,
        "OS": surv_data["OS"],
        "event": surv_data["Status"],
    }, index=surv_data.index)
    tmp = tmp.dropna()
    if tmp.shape[0] < 5 or tmp["event"].nunique() < 2 or tmp["OS"].nunique() < 2:
        continue
    try:
        cph.fit(tmp, duration_col="OS", event_col="event")
        hr = cph.hazard_ratios_.get("expr", None)
        if hr is None:
            coef = cph.params_.get("expr", None)
            if coef is not None:
                weights[gene] = coef
        else:
            weights[gene] = np.log(hr)
    except Exception as e:
        print(f"Skipping gene {gene} due to CoxPH error: {e}")

if len(weights) == 0:
    raise RuntimeError("No gene returned a valid Cox model. Check input data and filtering.")

weights_series = pd.Series(weights)


# ---------- 9. Weighted Signature Score ----------
df_norm_weighted = df_norm.loc[df_norm.index.intersection(weights_series.index)].copy()
df_norm_weighted = df_norm_weighted.reindex(columns=surv_data.index)
scores = (df_norm_weighted.T @ weights_series).reindex(surv_data.index)
surv_data["score"] = scores


# ---------- 10. High/Low Groups ----------
median_score = surv_data["score"].median()
surv_data["group"] = np.where(surv_data["score"] > median_score, "high", "low")


# ---------- 11. Kaplan–Meier Survival ----------
high_idx = surv_data["group"] == "high"
low_idx = surv_data["group"] == "low"
if high_idx.sum() == 0 or low_idx.sum() == 0:
    raise RuntimeError("One of the groups (high/low) is empty. Cannot perform log-rank test.")

result = statistics.logrank_test(
    surv_data.loc[high_idx, "OS"],
    surv_data.loc[low_idx, "OS"],
    event_observed_A=surv_data.loc[high_idx, "Status"],
    event_observed_B=surv_data.loc[low_idx, "Status"],
)
p_value = result.p_value
print(f"Log-rank test p-value: {p_value:.4g}")

kmf = KaplanMeierFitter()
plt.figure(figsize=(7, 6))
for g in ["high", "low"]:
    idx = surv_data["group"] == g
    if idx.sum() == 0:
        continue
    kmf.fit(surv_data.loc[idx, "OS"], event_observed=surv_data.loc[idx, "Status"], label=g)
    kmf.plot_survival_function(ci_show=False)

plt.title(f"Overall Survival (Log-rank p = {p_value:.4g})")
plt.xlabel("Time (OS)")
plt.ylabel("Survival Probability")
plt.grid(True)
plt.legend(title="Group")
plt.tight_layout()
plt.show()


# ---------- 12. Save Output ----------
surv_data.to_csv("surv_data.csv", index=True)
df_subset.to_csv("exp_data.csv")
weights_series.to_csv("gene_weights.csv")
print("Saved: surv_data.csv, exp_data.csv, gene_weights.csv")
